In [1]:
import os
import torch
import ast
import re
import warnings
import hashlib

import pandas as pd
import numpy as np
import networkx as nx

from collections import defaultdict
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

C:\Users\roans\anaconda3.0\envs\transformers\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Embedding using: {device}")

# Move Model to GPU once
tokenizer = AutoTokenizer.from_pretrained("../pipeline/dajobbert-kg-specialized")
embed_model = AutoModel.from_pretrained("../pipeline/dajobbert-kg-specialized").to(device)
embed_model.eval()

Embedding using: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31748, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [3]:
SCHEMA_NODES = [
    "job_title", "skill", "quality", "experience_level", 
    "work_experience", "education_level", "certification", 
    "location", "contract_type", "industry", "language", 
    "company", "salary", "candidate", "vacancy", "miscellaneous"
]

SCHEMA_TRIPLETS = [
    ('candidate', 'gets_recommended', 'vacancy'),
    ('candidate', 'has_skill', 'skill'),
    ('candidate', 'has_experience', 'work_experience'),
    ('candidate', 'has_education', 'education_level'),
    ('candidate', 'lives_in', 'location'),
    ('candidate', 'speaks_language', 'language'),
    ('candidate', 'has_job_title', 'job_title'),
    ('vacancy', 'requires_skill', 'skill'),
    ('vacancy', 'requires_quality', 'quality'),
    ('vacancy', 'requires_education', 'education_level'),
    ('vacancy', 'requires_experience_level', 'experience_level'),
    ('vacancy', 'has_location', 'location'),
    ('vacancy', 'offers_position', 'job_title'),
    ('vacancy', 'has_salary_range', 'salary'),
    ('job_title', 'is_in_industry', 'industry'),
    ('company', 'offers_position', 'vacancy'),
    ('candidate', 'related_to', 'miscellaneous'),
    ('vacancy', 'related_to', 'miscellaneous')
]
for ntype in SCHEMA_NODES: SCHEMA_TRIPLETS.append((ntype, 'self_loop', ntype))

def deep_clean(name):
    name = str(name).lower()
    name = re.sub(r'jie:|xsd:|_jie_|_jip_', '', name)
    name = re.sub(r'_\d+$', '', name)
    return name.strip('_').replace('_', ' ').strip()

def get_ntype(node_str):
    raw = node_str.upper()
    if "CANDIDATE" in raw: return "candidate"
    if "POSITION" in raw or "VACANCY" in raw: return "vacancy"
    for a in SCHEMA_NODES:
        if a.upper().replace('_', ' ') in raw: return a
    return "miscellaneous"

def build_embedding_cache(hits_misses, batch_size=64):
    """Embeds all unique nodes in one massive GPU-accelerated batch."""
    unique_names = set()
    for graphs in hits_misses.values():
        for g in graphs.values():
            for node in g.nodes():
                c_n = deep_clean(node)
                if not (c_n.isdigit() or c_n in ['decimal', 'integer', 'string', '']):
                    unique_names.add(c_n)
    
    name_list = list(unique_names)
    cache = {}
    
    for i in range(0, len(name_list), batch_size):
        batch_names = name_list[i : i + batch_size]
        inputs = tokenizer(batch_names, return_tensors="pt", padding=True, 
                           truncation=True, max_length=64).to(device)
        with torch.no_grad():
            outputs = embed_model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :].cpu()
        for name, emb in zip(batch_names, embeddings):
            cache[name] = emb
            
    return cache

In [4]:
def deep_clean(name):
    name = str(name).lower()
    name = re.sub(r'jie:|xsd:|_jie_|_jip_', '', name)
    name = re.sub(r'_\d+$', '', name)
    return name.strip('_').replace('_', ' ').strip()

def get_ntype(node_str):
    raw = node_str.upper()
    if "CANDIDATE" in raw: return "candidate"
    if "POSITION" in raw or "VACANCY" in raw: return "vacancy"
    for a in SCHEMA_NODES:
        if a.upper().replace('_', ' ') in raw: return a
    return "miscellaneous"


def get_split(user_id, train_size=0.8, val_size=0.1):
    """
    Deterministically assigns a user to train/val/test based on their ID hash.
    Guarantees a candidate never leaks across sets across different model runs.
    """
    # Create a consistent integer from the user_id string
    hash_val = int(hashlib.md5(str(user_id).encode('utf-8')).hexdigest(), 16)
    bucket = (hash_val % 1000) / 1000.0  # Float between 0.0 and 1.0
    
    if bucket < train_size:
        return "train"
    elif bucket < (train_size + val_size):
        return "val"
    else:
        return "test"

def create_dataloaders(hits_misses, truth_dict, embedding_cache, 
                       train_size=0.8, val_size=0.1):
    
    embedding_dim = embed_model.config.hidden_size
    train_list, val_list, test_list = [], [], []
    
    # Sort keys just to guarantee absolute identical iteration order
    u_keys = sorted(list(hits_misses.keys())) 

    for i, user_id in tqdm(enumerate(u_keys), total=len(u_keys)):
        
        # --- THE FIX: Assign target list using the deterministic hash ---
        split = get_split(user_id, train_size, val_size)
        if split == "train":
            target_list = train_list
        elif split == "val":
            target_list = val_list
        else:
            target_list = test_list
        # ---------------------------------------------------------------
            
        graphs = hits_misses[user_id]
        
        temp_graphs, truth_values = [], []
        node_info, head_ids, tail_ids = {}, [], []
        global_node_counter = 1

        for sg_idx, (tail_id, G_raw) in enumerate(graphs.items()):
            if not G_raw: continue
            
            # Find Anchors
            cand_node, vac_node = None, None
            for n in list(G_raw.nodes()):
                c_n = deep_clean(n)
                if str(user_id) in c_n and "candidate" in c_n: cand_node = n
                if str(tail_id) in c_n and ("position" in c_n or "vacancy" in c_n): vac_node = n

            if cand_node and vac_node:
                G_raw.add_edge(cand_node, vac_node, edge_type="gets_recommended")
                undirected = G_raw.to_undirected()
                if nx.has_path(undirected, cand_node, vac_node):
                    main_comp = nx.node_connected_component(undirected, cand_node)
                    G_raw = G_raw.subgraph(main_comp).copy()
                else: continue

            valid_nodes_in_sg = []
            for n in G_raw.nodes():
                c_n = deep_clean(n)
                # Look up from CACHE instead of running BERT
                if c_n in embedding_cache:
                    ntype = get_ntype(n)
                    new_name = f"{n}_{i}_{sg_idx}"
                    node_info[new_name] = (ntype, global_node_counter, sg_idx)
                    
                    G_raw.nodes[n]['x'] = embedding_cache[c_n]
                    
                    if ntype == "candidate" and str(user_id) in c_n:
                        head_ids.append(global_node_counter)
                    elif (ntype == "vacancy" or ntype == "job_title") and str(tail_id) in c_n:
                        tail_ids.append(global_node_counter)
                    
                    valid_nodes_in_sg.append((n, new_name))
                    global_node_counter += 1

            if valid_nodes_in_sg:
                mapping = {old: new for old, new in valid_nodes_in_sg}
                sg_final = G_raw.subgraph([n for n, _ in valid_nodes_in_sg])
                temp_graphs.append(nx.relabel_nodes(sg_final, mapping))
                
                u_truth = truth_dict.get(user_id, truth_dict.get(str(user_id), {}))
                label = u_truth.get(tail_id, u_truth.get(int(tail_id), 0))
                truth_values.append(float(label))

        if len(temp_graphs) == 0:
            continue

        # Build HeteroData
        merged_G = nx.compose_all(temp_graphs)
        data = HeteroData()
        node_map = {} 

        type_data = defaultdict(lambda: {"x":[], "u":[], "s":[]})
        for node in merged_G.nodes():
            ntype, uid, sg = node_info[node]
            node_map[node] = len(type_data[ntype]["x"])
            type_data[ntype]["x"].append(merged_G.nodes[node]['x'])
            type_data[ntype]["u"].append(uid)
            type_data[ntype]["s"].append(sg)

        for ntype in SCHEMA_NODES:
            if ntype in type_data:
                data[ntype].x = torch.stack(type_data[ntype]["x"])
                data[ntype].unique_node_id = torch.tensor(type_data[ntype]["u"])
                data[ntype].sub_graph = torch.tensor(type_data[ntype]["s"])
            else:
                data[ntype].x = torch.zeros((1, embedding_dim))
                data[ntype].unique_node_id = torch.tensor([0])
                data[ntype].sub_graph = torch.tensor([-1])

        # Edges and Self-loops
        for triplet in SCHEMA_TRIPLETS:
            data[triplet].edge_index = torch.empty((2, 0), dtype=torch.long)

        for u, v, d in merged_G.edges(data=True):
            u_t, _, _ = node_info[u]; v_t, _, _ = node_info[v]
            etype = deep_clean(d.get("edge_type", "related_to")).replace(' ', '_')
            triplet = (u_t, etype, v_t)
            if triplet in [ (t[0], t[1], t[2]) for t in SCHEMA_TRIPLETS]:
                new_e = torch.tensor([[node_map[u]], [node_map[v]]], dtype=torch.long)
                data[triplet].edge_index = torch.cat([data[triplet].edge_index, new_e], dim=1)

        for ntype in data.node_types:
            num_n = data[ntype].x.size(0)
            idx = torch.arange(num_n)
            data[ntype, "self_loop", ntype].edge_index = torch.stack([idx, idx], dim=0)

        data.y = torch.tensor(truth_values)
        data.head_nodes = torch.tensor(head_ids)
        data.tail_nodes = torch.tensor(tail_ids)
        
        # Finally, append to the mathematically assigned list
        target_list.append(data)
        
    return train_list, val_list, test_list

In [6]:
os.makedirs("../dataloaders", exist_ok=True)

for isco in [True, False]:
    for model in ["qwen", "gemma", "llama"]:
        for prompt in ["structured", "semi-structured", "unstructured"]:
            
            # 1. Determine filenames
            suffix = "_isco" if isco else ""
            file_name = f"subgraphs_{model}_{prompt}{suffix}.xlsx"
            file_path = f"../kg_construction/{file_name}"
            
            if not os.path.exists(file_path):
                print(f"Skipping {file_name} (Not found)")
                continue
                
            print(f"\n{'='*50}\nProcessing: {file_name}\n{'='*50}")
            
            # 2. Load and filter Data
            df_graphs = pd.read_excel(file_path)
            df_graphs["response"] = df_graphs["response"].apply(lambda x: x if x == 0 else 1)
            
            non_zero = df_graphs.groupby("cvid")["response"].sum() > 0
            non_zero = set(non_zero[non_zero == True].index)
            df_graphs = df_graphs[df_graphs["cvid"].isin(non_zero)]
            
            # (Fixed pandas include_groups warning)
            truth_dict = df_graphs.groupby('cvid').apply(
                lambda x: dict(zip(x['vacancy'], x['response'])),
                include_groups=False 
            ).to_dict()

            # 3. Parse Subgraphs (Fixed positional warnings by using column names)
            hits = defaultdict(dict)
            misses = defaultdict(dict)
            
            for _, row in tqdm(df_graphs.iterrows(), total=len(df_graphs), desc=f"Parsing JSON ({model})"):
                if not pd.isna(row["graph"]):    
                    data_dict = ast.literal_eval(row["graph"])
                    g = nx.node_link_graph(data_dict)
                
                    if row["response"] == 0:
                        misses[row["cvid"]][row["vacancy"]] = g
                    else:
                        hits[row["cvid"]][row["vacancy"]] = g
                        
            # Merge hits and misses for valid users
            hits_misses = {}
            valid_users = set(hits.keys()).intersection(set(misses.keys()))
            for user in valid_users:
                hits_misses[user] = {**hits[user], **misses[user]}

            if len(hits_misses) == 0:
                print(f"No valid hits/misses overlap for {file_name}. Skipping dataloader creation.")
                continue

            # 4. Embed and build DataLoaders
            print("Building cache...")
            cache = build_embedding_cache(hits_misses)
            
            print("Creating HeteroData...")
            train_loader, val_loader, test_loader = create_dataloaders(hits_misses, truth_dict, cache)
            
            trainloader = DataLoader(train_loader) 
            valloader = DataLoader(val_loader) 
            testloader = DataLoader(test_loader)
            
            # 5. Save dynamically named DataLoaders
            train_path = f"../dataloaders/graph_trainloader_{model}_{prompt}{suffix}.pth"
            val_path = f"../dataloaders/graph_valloader_{model}_{prompt}{suffix}.pth"
            test_path = f"../dataloaders/graph_testloader_{model}_{prompt}{suffix}.pth"
            
            torch.save(trainloader, train_path)
            torch.save(valloader, val_path)
            torch.save(testloader, test_path)
            print(f"Saved dataloaders for {model}_{prompt}{suffix}!") 


Processing: subgraphs_qwen_structured_isco.xlsx


Parsing JSON (qwen):   0%|          | 0/13335 [00:00<?, ?it/s]

Building cache...
Creating HeteroData...


  0%|          | 0/579 [00:00<?, ?it/s]

469 61 49
Saved dataloaders for qwen_structured_isco!

Processing: subgraphs_qwen_semi-structured_isco.xlsx


...

447 57 50
Saved dataloaders for llama_semi-structured!

Processing: subgraphs_llama_unstructured.xlsx


Parsing JSON (llama):   0%|          | 0/13335 [00:00<?, ?it/s]

Building cache...
Creating HeteroData...


  0%|          | 0/547 [00:00<?, ?it/s]

445 54 48
Saved dataloaders for llama_unstructured!
